In [1]:
import pandas as pd
df = pd.read_csv("../data/raw_churn_data.csv")
df.shape

(10000, 12)

In [ ]:
df.columns.tolist()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df['customer_id'].duplicated().sum()

In [ ]:
df['churn'].value_counts()

In [ ]:
df['churn'].value_counts(normalize=True)

In [ ]:
df.groupby('country')['churn'].mean()

In [ ]:
df.groupby('gender')['churn'].mean()

In [ ]:
df.groupby('churn')[['credit_score', 'age', 'balance', 'estimated_salary']].mean()

In [ ]:
df.groupby('churn')[['tenure', 'products_number', 'credit_card', 'active_member']].mean()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(data=df, x='age', hue='churn', kde=True, bins=30)
plt.show()

In [ ]:
sns.histplot(data=df, x='balance', hue='churn', kde=True, bins=30)
plt.show()

In [ ]:
df['has_zero_balance'] = df['balance'] == 0
df.groupby('has_zero_balance')['churn'].mean()

In [ ]:
df.corr(numeric_only=True)['churn'].sort_values(ascending=False)

In [ ]:
df.groupby('products_number')['churn'].mean()

In [ ]:
df['products_number'].value_counts()

In [ ]:
df.groupby('tenure')['churn'].mean()

In [ ]:
df['credit_score_bucket'] = pd.cut(df['credit_score'], bins=5)
df.groupby('credit_score_bucket')['churn'].mean()

## EDA Summary

**Dataset:** 10,000 customers, 12 columns, no missing values, no duplicates.

**Target variable:** `churn` - 20.4% churned, 79.6% stayed (moderate class imbalance; accuracy alone won't be a meaningful metric for this problem).

### Strong predictors

- **`age`** - churners average ~45 years old vs ~37 for non-churners. Histogram confirms a smooth, genuine upward shift in churn risk with age (not a spike or artifact).
- **`products_number`** - highly non-linear relationship, missed almost entirely by the raw correlation coefficient (-0.048, near zero). Actual churn rates by product count: 1 product = 27.7%, 2 products = 7.6% (safest group), 3 products = 82.7%, 4 products = 100% (though only 60 customers hold 4 products, so treat that exact figure cautiously, the direction is still a strong, real signal).
- **`country`** - Germany churns at ~32.4%, roughly double France (16.2%) and Spain (16.7%).
- **`active_member`** - active members churn less often (36.1% of churners were inactive vs only 44.5% active... simplify:) active members churn at a lower rate than inactive members (active-member rate: 55% among stayers vs 36% among churners).
- **`balance`** - not a simple linear effect. Distribution is bimodal: a large cluster of customers with exactly zero balance (low churn, 13.8%) and a separate cluster of funded customers (balance 50k+, higher churn, 24.1% among the non-zero group). Raw balance correlation understated this because it blends two different populations.
- **`gender`** - Female customers churn at ~25.1% vs ~16.5% for Male customers.

### Weak predictors

- **`tenure`** - flat across all values (17–23% churn range, no trend). Confirmed via groupby, not just correlation.
- **`credit_card`** - no meaningful difference (0.71 vs 0.70 average ownership between groups).
- **`estimated_salary`** - no relationship (correlation ~0.01).
- **`credit_score`** - mostly flat (~19–21% churn) except the lowest bucket (below ~450), which churns at 32.3% - a smaller, less certain sample, but a plausible risk-flag pattern.

### Implications for Phase 3

- `products_number`, `balance`, and `credit_score` all show effects that raw numeric values or linear correlation would miss or understate, feature engineering (e.g. a `has_zero_balance` flag, bucketing `products_number` and `credit_score`) is likely to matter more than using these as raw continuous/numeric inputs.
- `country` and `gender` will need categorical encoding before modeling.
- `tenure`, `credit_card`, and `estimated_salary` are candidates for exclusion or de-prioritization, pending confirmation once we move into actual modeling (Phase 4) - EDA suggests weak signal, but the final feature set should be validated with the model itself, not decided on EDA alone.

In [ ]:
import sys
sys.path.append('..')

from src.preprocessing import drop_unused_columns

df_test = drop_unused_columns(df)
df_test.columns.tolist()

In [ ]:
from src.preprocessing import add_zero_balance_flag

df_test2 = add_zero_balance_flag(df)
df_test2[['balance', 'has_zero_balance']].head()

In [ ]:
from src.preprocessing import encode_categoricals

df_test3 = encode_categoricals(df)
df_test3.columns.tolist()

In [ ]:
from src.preprocessing import split_data

X_train, X_test, y_train, y_test = split_data(df)
X_train.shape, X_test.shape

In [ ]:
from src.preprocessing import drop_unused_columns

df_test = drop_unused_columns(df)
df_test.columns.tolist()

In [ ]:
y_train.value_counts(normalize=True)

In [ ]:
y_test.value_counts(normalize=True)

In [ ]:
import sys
sys.path.append('..')

ModuleNotFoundError: No module named 'src'